In [1]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [4]:
from utils.generic_utils import SQLModule
from utils.data_utils import StockPriceProcess

In [5]:
# Insert stock codes here
STOCK_CODES = ['YAL', 'TPG', 'TNE']
PLAN_RANGE = 365

In [7]:
engine = SQLModule.get_engine(country = 'australia')
end_date = datetime.today().date()
start_date = end_date - relativedelta(days = PLAN_RANGE)
# Some metrics are not filter due to comparative comparison
stock_query = f"""
    SELECT
        stock_code,
        date,
        close
    FROM transaction
    WHERE
        stock_code IN {tuple(STOCK_CODES)}
        AND
        date >= DATE '{start_date}'
        AND
        date <= DATE '{end_date}'
    ORDER BY date
"""
df = pd.read_sql_query(stock_query, engine)
df = StockPriceProcess.frame_var(df)
df = StockPriceProcess.remove_invalid_data(df, country = 'australia')
df

C:\Users\khoan\OneDrive\Documents\stock_data_scraper\utils\data_utils.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  no_holiday_df[col] = no_holiday_df[col].fillna(method = 'bfill').fillna(method = 'ffill')
C:\Users\khoan\OneDrive\Documents\stock_data_scraper\utils\data_utils.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_holiday_df[col] = no_holiday_df[col].fillna(method = 'bfill').fillna(method = 'ffill')


,TNE,TPG,YAL
date,,,
2024-02-12,16.200018,5.148895,5.410682
2024-02-13,16.090960,5.158519,5.353927
2024-02-14,15.833187,5.071902,5.335008
2024-02-15,16.348732,5.168143,5.193120
2024-02-16,16.447876,5.168143,5.278253
...,...,...,...
2025-02-04,31.270000,4.320000,6.550000
2025-02-05,31.799999,4.400000,6.550000
2025-02-06,31.980000,4.430000,6.580000


In [8]:
fig = make_subplots(rows = len(df.columns), cols = 1, subplot_titles = df.columns)
fig.update_annotations(font = dict(size = 20))
for num,stock_code in enumerate(df.columns):
    fig.add_trace(
        go.Scatter(
            x = df.index, 
            y = df[stock_code],  
            name = stock_code,
            marker = dict(color = 'blue'),
            showlegend = False
        ),
        row = num + 1, col = 1
    )
fig.update_yaxes(title = 'Price', showgrid = True, gridcolor = 'gray')
fig.update_xaxes(title = 'Date')
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    height = 400 * len(df.columns),
    font = dict(size = 20)
)
fig.show()

In [9]:
# Plot box
fig = go.Figure()

for stock in df.columns:
    fig.add_trace(go.Box(y = df[stock], name = stock, showlegend = False, marker = dict(color = 'blue')))
fig.update_yaxes(title = 'Price', showgrid = True, gridcolor = 'gray')
fig.update_xaxes(title = 'Stocks')
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    width = 200 * len(df.columns),
    height = 600,
    font = dict(size = 20)
)
fig.show()

In [10]:
# Plot box
fig = go.Figure()

for stock in df.columns:
    fig.add_trace(go.Box(y = df[stock].pct_change(), name = stock, showlegend = False, marker = dict(color = 'blue')))
fig.update_yaxes(title = 'Price', showgrid = True, gridcolor = 'gray')
fig.update_xaxes(title = 'Stocks')
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    width = 200 * len(df.columns),
    height = 600,
    font = dict(size = 20)
)
fig.show()

In [ ]:
pct_change = df.pct_change().dropna()
# Calculate cumulative returns
cumulative_returns = (1 + pct_change).cumprod()
fig = go.Figure()
for col in cumulative_returns.columns:
    fig.add_trace(
        go.Scatter(
            x = cumulative_returns.index, 
            y = cumulative_returns[col],  
            name = col
        )
    )
fig.update_yaxes(title = 'Cumulative returns', showgrid = True, gridcolor = 'gray', tickformat=".0%")
fig.update_xaxes(title = 'Date')
fig.update_layout(
    plot_bgcolor = 'white',
    paper_bgcolor = 'white',
    height = 600,
    font = dict(size = 20)
)
fig.show()